# A4 · QC del cubo (M1–M5)

**Spec:** [`docs/spec_A4_codex_cube_qc.md`](../docs/spec_A4_codex_cube_qc.md)  |  **Bloque:** A · Reducción  |  **Run de este set:** `ROXs12b_realigned`

Métricas de calidad del cubo: solución en λ (M1/M2), flujo absoluto (M3), STAT (M5).

| | |
|---|---|
| **Entrada** | `cube_telcorr.fits`, SKY_SPECTRUM, Gaia DR3 |
| **Salida (QC/productos)** | `stages/stage00q_qc.json` |
| **Consume aguas abajo** | D2/E1 (usan σ empírico), E3 (flujo) |


## Qué mide A4: las 5 métricas de calidad del cubo (M1–M5)

A4 no re-reduce: **verifica la calibración** del cubo con 5 métricas, cada una un aspecto distinto.

Los valores de la tabla son los de **este objeto**, resueltos de su `stage00q_qc.json` al generar el notebook (`n/d` = métrica no medida todavía para esta cadena).

| Métrica | Qué mide | Resultado | Significado |
|---|---|---|---|
| **M1** | Exactitud de la solución de λ (offset vs airglow) | **green** (0.074 Å) | residuo de la solución en λ del cubo |
| **M2** | LSF (ancho de la función de dispersión) | **yellow** (2.383 Å @Hα) | resolución espectral real, medida del airglow y comparada con la LSF publicada de MUSE (Bacon+2017); es la medida la que se usa en E1/E3/G2 |
| **M3** | Calibración de flujo absoluto (vs Gaia RP) | **green** (factor 0.973) | cuánto se aparta la escala de flujo de la fotometría Gaia |
| **M4** | Residuo de cielo (la `R` de A2) | **yellow** (R 0.547) | calidad de la sustracción de cielo (ver A2) |
| **M5** | Fiabilidad del STAT (varianza del cubo) | **red** (4.26×) | cuánto subestima el STAT el ruido → si es alto, σ **siempre** empírico |

Las dos decisiones grandes de A4: **M3** (¿la escala de flujo es utilizable?) y **M5** (¿sirve el STAT como σ, o rige la regla *control = objeto*, [`docs/noise_model.md`](../docs/noise_model.md)?).


## Cómo ejecutar de forma independiente

Etapa de **reducción**: la celda de abajo resuelve el comando real para **este objeto** a partir de su `chain.reduction_profile` y de su config, y puede lanzarlo. Son trabajos largos (ver coste), así que se lanzan en segundo plano con el log a la vista; el notebook no se bloquea.

Si algún dato no está declarado en el config del run, la celda lo dice y **no lanza** en vez de inventarse una ruta.

Comando histórico de referencia:

```bash
conda activate MUSE
# M1/M2 (LSF) desde el airglow cacheado:
python -m musepipe.qc.cube_qc m1m2-sky --sky-spectrum <SKY_SPECTRUM...> --qc-output <...>
# M3 (flujo absoluto vs Gaia RP, con growth-curve + truncación):
python -m musepipe.qc.cube_qc m3-flux --cube <cube_telcorr.fits> --run-id $RUN \
    --aperture-correction growth_curve --truncation-correction --qc-output <...>
```


In [ ]:
import os, sys
# Localiza la raíz del repo ascendiendo hasta encontrar `musepipe/` (robusto a
# la profundidad: funciona con el cwd en notebooks/<obj>/, en notebooks/ o en la
# raíz). Añade la raíz (para `import musepipe`) y notebooks/ (para `_nbcommon`).
_d = os.getcwd()
while _d != os.path.dirname(_d):
    if os.path.isdir(os.path.join(_d, 'musepipe')) and os.path.isdir(os.path.join(_d, 'notebooks')):
        break
    _d = os.path.dirname(_d)
_root = _d
for _p in (_root, os.path.join(_root, 'notebooks')):
    if _p not in sys.path:
        sys.path.insert(0, _p)
import _nbcommon as nb
try:
    import matplotlib as mpl
    mpl.rcParams['figure.dpi'] = 120     # retina dobla esto sin agrandar
    mpl.rcParams['savefig.dpi'] = 200
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats('retina')
except Exception:
    pass
RUN_ID = nb.resolve_run_id('ROXs12b_realigned')
print('run  =', RUN_ID)
print('root =', _root)
print('dir  =', nb.run_dir(RUN_ID))
print('QC   =', nb.provenance_line('stages/stage00q_qc.json', RUN_ID))


## Ejecutar o auditar


In [ ]:
cmd, target_run, missing = nb.launch_command('A4', RUN_ID)
print('run que ejecuta esta etapa:', target_run)
print('comando resuelto para este objeto:')
print('   ', cmd or '(sin plantilla)')
if missing:
    print()
    print('NO se puede lanzar: faltan datos en el config del run.')
    print('   sin resolver:', ', '.join(missing))
    print(f'   declara esas claves en runs/{target_run}/config/config.json')

RUN = False   # -> True para LANZAR (trabajo largo: revisa el coste arriba)

if RUN and not missing:
    import subprocess, time
    from pathlib import Path
    log = Path(nb.run_dir(target_run)) / 'logs' / f'a4_launch.log'
    log.parent.mkdir(parents=True, exist_ok=True)
    with open(log, 'w') as fh:
        proc = subprocess.Popen(cmd, shell=True, cwd=str(nb.project_root()),
                                stdout=fh, stderr=subprocess.STDOUT)
    print(f'lanzado en segundo plano (pid {proc.pid}); log -> {log}')
    print('sigue el progreso con:  !tail -f', log)
elif RUN:
    print('RUN=True pero hay datos sin resolver: no se lanza nada.')
else:
    print()
    print('Modo auditoría (RUN=False): abajo se carga el QC existente.')


## QC / resultados


In [ ]:
qc = nb.load_qc_optional('stages/stage00q_qc.json', RUN_ID)
nb.show(qc, keys=['m1_wavelength.status', 'm2_lsf.status', 'm3_flux.status', 'm4_sky.status', 'm5_stat.status'], title='A4')


## Resultados que llevaron a la conclusión

Resumen M1–M5 del `stage00q_qc.json` (estado + cifra de cabecera).


In [ ]:
if qc is None:
    print('(evidencia omitida: la etapa no se ha ejecutado para esta cadena)')
else:
    with nb.evidence_guard('A4', 'stages/stage00q_qc.json'):
        q = nb.load_qc('stages/stage00q_qc.json', RUN_ID)
        m1, m2, m3, m4, m5 = (q['m1_wavelength'], q['m2_lsf'], q['m3_flux'], q['m4_sky'], q['m5_stat'])
        def _f(v, fmt='.3f'):
            """Formatea, o 'n/d' si la métrica no se midió en esta cadena."""
            return format(v, fmt) if isinstance(v, (int, float)) else 'n/d'
        rows = [
            ('M1 λ-solution', m1.get('status'), f"offset {_f(m1.get('offset_median_A'))} Å (±{_f(m1.get('offset_err_A'))}), {m1.get('n_lines')} líneas"),
            ('M2 LSF',        m2.get('status'), f"{_f(m2.get('lsf_fwhm_at_halpha_A'))} Å @Hα, dev máx vs referencia {_f(m2.get('max_dev_vs_nominal_pct'), '.1f')}% [{m2.get('nominal_reference', 'referencia no declarada en el QC')}]"),
            ('M3 flujo abs',  m3.get('status'), f"factor {_f(m3.get('flux_factor'))} vs Gaia {m3.get('band')} (growth-curve r={_f(m3.get('plateau_radius_px'), '.0f')})"),
            ('M4 cielo',      m4.get('status'), f"R = {m4.get('R')}"),
            ('M5 STAT',       m5.get('status'), f"factor spaxel {m5.get('factor_spaxel_median')}× (cuánto subestima el STAT el ruido)"),
        ]
        for name, st, detail in rows:
            print(f'{name:15s} [{str(st):9s}] {detail}')


## Plot 1 — M2 LSF (medida vs referencia publicada) y M3 growth-curve

Ambos desde el QC (baratos, sin cubo).

**Izq:** LSF medida del airglow (azul) frente a la **LSF de referencia de MUSE publicada**: FWHM(λ) = 5.866·10⁻⁸ λ² − 9.187·10⁻⁴ λ + 6.040 Å ([Bacon et al. 2017, A&A 608, A1](https://doi.org/10.1051/0004-6361/201730833), Ec. 8 — mediana de la LSF medida en los cubos del MUSE UDF, dispersión 1–3%). Sustituye a la interpolación lineal en R (1770@4800 Å → 3590@9300 Å) que se usaba antes y que no procedía de ninguna publicación. Salvedad: la referencia es de WFM, así que sirve como **patrón de comparación**, no como la LSF de este cubo — aguas abajo (E1/E3/G2) se usa siempre la **medida**. Para este objeto: medida @Hα = 2.383 Å (referencia @Hα = 2.537 Å).

**Der:** el flujo en banda RP crece con el radio hasta el *plateau* (halo AO capturado) → `flux_factor = 0.973`.


In [ ]:
try:
    import numpy as np
    import matplotlib.pyplot as plt
    q = nb.load_qc('stages/stage00q_qc.json', RUN_ID)
    m2 = q['m2_lsf']; tab = m2['table_A_fwhm']
    w = np.array([r['wave_A'] for r in tab]); f = np.array([r['fwhm_A'] for r in tab])
    # Curva de referencia: se RECALCULA con la función canónica en vez de leer
    # `nominal_fwhm_A` del QC, porque un QC escrito antes del cambio de
    # referencia llevaría todavía la curva antigua (interpolación en R).
    try:
        from musepipe.qc.cube_qc import nominal_muse_fwhm_A as ref_fn
        from musepipe.qc.cube_qc import MUSE_LSF_REFERENCE as ref_cite
        from musepipe.qc.cube_qc import MUSE_LSF_REFERENCE_SHORT as ref_short
    except ImportError:
        o = np.argsort(w); _nom = np.array([r['nominal_fwhm_A'] for r in tab])
        ref_fn = lambda x: np.interp(np.asarray(x, float), w[o], _nom[o])
        ref_cite = m2.get('nominal_reference', 'referencia guardada en el QC')
        ref_short = 'QC'
        print('(sin musepipe en este kernel: uso la referencia guardada en el QC)')
    ref = np.asarray(ref_fn(w), dtype=float)
    ref_ha = float(np.atleast_1d(ref_fn([6563.0]))[0])
    dev = np.abs(f / ref - 1.0) * 100.0
    m3 = q['m3_flux']; gc = m3['growth_curve']
    gr = np.array([p['radius_px'] for p in gc]); gf = np.array([p['band_flux'] for p in gc])

    fig, (axL, axR) = plt.subplots(1, 2, figsize=(13, 4.2))
    axL.scatter(w, f, s=10, color='tab:blue', label='LSF medida (airglow)')
    wg = np.linspace(float(w.min()), float(w.max()), 300)
    axL.plot(wg, ref_fn(wg), color='0.35', lw=1.6,
             label=f'LSF de referencia ({ref_short})')
    axL.axvline(6563, color='tab:red', ls=':', label='Hα')
    hal = m2.get('lsf_fwhm_at_halpha_A')
    if hal: axL.axhline(hal, color='tab:red', ls='--', lw=1)
    axL.set_xlabel('λ [Å]'); axL.set_ylabel('FWHM LSF [Å]')
    axL.set_title(f"M2 · LSF medida vs referencia ({m2['status']})\n"
                  f"@Hα = {hal:.3f} Å vs {ref_ha:.3f} Å (ref) · dev máx {dev.max():.1f}%",
                  fontsize=10)
    axL.text(0.02, 0.035, f'Referencia: {ref_cite}', transform=axL.transAxes,
             fontsize=7, color='0.35',
             bbox=dict(facecolor='white', alpha=0.75, edgecolor='none', pad=1.5))
    axL.legend(fontsize=8, loc='upper right')
    # Procedencia de la referencia: avisa si el QC en disco se escribió con otra.
    stored_cite, stored_dev = m2.get('nominal_reference'), m2.get('max_dev_vs_nominal_pct')
    if stored_cite != ref_cite:
        print(f'AVISO: el QC en disco declara referencia {stored_cite!r} y este plot usa '
              f'{ref_cite!r}.')
        if isinstance(stored_dev, (int, float)):
            print(f'       dev máx: {dev.max():.1f}% (recalculada aquí) vs '
                  f'{stored_dev:.1f}% (guardada). El estado M2 del QC se calculó con la '
                  'referencia antigua.')
        print('       Re-ejecuta A4 (m1m2-sky) para regenerar el QC con la referencia actual.')
    axR.plot(gr, gf, 'o-', color='tab:green')
    axR.axvline(m3['plateau_radius_px'], color='0.5', ls='--',
                label=f"plateau r={m3['plateau_radius_px']:.0f}px")
    axR.set_xlabel('radio de apertura [px]'); axR.set_ylabel('flujo en banda RP')
    axR.set_title(f"M3 · growth-curve → factor flujo={m3['flux_factor']:.3f} ({m3['status']})")
    axR.legend(fontsize=8); fig.tight_layout()
    outdir = nb.run_dir(RUN_ID) / 'plots' / 'a4_qc'; outdir.mkdir(parents=True, exist_ok=True)
    fig.savefig(outdir / 'm2_lsf_m3_growth.png', dpi=110)
    print('figura ->', outdir / 'm2_lsf_m3_growth.png'); plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


## Plot 2 — M5: por qué el STAT no sirve (covarianza del remuestreo)

M5 mide cuánto subestima el STAT el ruido por spaxel (**4.26×** en este objeto). El QC solo guarda esa mediana, así que ilustro el **mecanismo** con la inflación espacial de G1 (almacenada): al sumar en cajas N×N la varianza real se infla frente a la suma ingenua de STAT (que asume píxeles independientes, =1) hasta ~19× en 5×5. Es la correlación introducida por el remuestreo del cubo → **σ siempre empírico, control = objeto**.

> Dependencia: este plot lee `stages/stage_g1_qc.json` (etapa G1); si G1 no ha corrido en el run, la celda degrada a un mensaje.


In [ ]:
try:
    import matplotlib.pyplot as plt
    q = nb.load_qc('stages/stage00q_qc.json', RUN_ID)
    m5 = q['m5_stat']
    g1 = nb.load_qc('stages/stage_g1_qc.json', RUN_ID)['covariance']
    infl = g1['spatial_inflation_by_box']
    boxes = sorted(infl, key=lambda k: int(k))
    xs = [f'{int(b)}×{int(b)}' for b in boxes]; vals = [infl[b] for b in boxes]

    fig, ax = plt.subplots(figsize=(8.5, 4.3))
    ax.bar(xs, vals, color='tab:orange', alpha=0.85)
    for i, v in enumerate(vals):
        ax.text(i, v + 0.3, f'{v:.1f}×', ha='center', fontsize=9)
    ax.axhline(1.0, color='tab:green', ls='--',
               label='STAT asume =1 (píxeles independientes)')
    ax.set_xlabel('caja de integración (N×N spaxels)')
    ax.set_ylabel('inflación varianza real / suma ingenua')
    fac = m5.get('factor_spaxel_median')
    fac_txt = f"STAT ~{fac}× bajo por spaxel" if isinstance(fac, (int, float)) \
        else f"M5 no medida en esta cadena ({m5.get('status')})"
    ax.set_title(f"M5 [{m5['status']}]: {fac_txt} "
                 f"(+ covarianza del remuestreo en apertura)")
    ax.legend(fontsize=8); fig.tight_layout()
    outdir = nb.run_dir(RUN_ID) / 'plots' / 'a4_qc'; outdir.mkdir(parents=True, exist_ok=True)
    fig.savefig(outdir / 'm5_stat_inflation.png', dpi=110)
    print('figura ->', outdir / 'm5_stat_inflation.png'); plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


## Decisiones y notas
- **M3 [green]**: flujo absoluto contrastado con Gaia DR3 RP, factor 0.973 tras growth-curve + truncación de cola.
- **M5 [red]**: factor de subestimación del STAT = 4.26× por spaxel (covarianza del remuestreo: inherente, no un defecto del cubo) → σ SIEMPRE empírico, control=objeto. Limitación aceptada en F1. · [`docs/noise_model.md`](../docs/noise_model.md)
- **M2 LSF@Hα = 2.383 Å medido** del airglow, frente a los 2.537 Å de la referencia publicada (Bacon et al. 2017, A&A 608, A1, Ec. 8); en E1/E3/G2 se usa la MEDIDA, nunca la referencia.


## Conclusión (registrada)

**A4: cubo caracterizado.** Estado por métrica para **este objeto** (resuelto del `stage00q_qc.json` de su cadena al generar el notebook; `n/d` = aún no medida). La procedencia exacta del QC la imprime la celda de setup.

- **M1 [green]:** offset 0.074 Å sobre 72 líneas de airglow (solución de λ).
- **M2 [yellow]:** LSF 2.383 Å @Hα medida del airglow (referencia publicada: 2.537 Å @Hα, Bacon+2017 Ec. 8); es la LSF **medida** la que se usa en E1/E3/G2.
- **M3 [green]:** flujo absoluto vs Gaia DR3 RP, factor 0.973, con growth-curve (halo AO) + truncación de cola.
- **M4 [yellow]:** residuo de cielo R=0.547 (ver A2).
- **M5 [red]:** factor de subestimación del STAT = 4.26× por spaxel, por la covarianza del remuestreo (inherente, no defecto) → σ SIEMPRE empírico (control=objeto). Limitación aceptada en F1.
- **Impacto:** M5 fija la regla de ruido de toda la cadena (D2/E1/E3); M3 sostiene el flujo absoluto de E3.
